## LLM Consistency Testing with DeMorgan's Law Mutations

This notebook contains code for testing code inconsistency using DeMorgan's law mutations on boolean expressions

In [ ]:
import os
import sys
import ast
import pandas as pd

In [ ]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [ ]:
from llm_models.code_llms import Mistral
from code_inconsistency.code_inconsistency_tester import LLMConsistencyTester
from code_inconsistency.prompt_templates.prompt_template import CodeInconsistencyPromptTemplate
from code_inconsistency.demorgan_consistency_tester import DeMorganConsistencyTester

In [ ]:
class DeMorganPreConditionFilter:
    """
    Helper class to filter database entries that contain boolean operations 
    suitable for DeMorgan's law transformations.
    """
    
    @staticmethod
    def has_boolean_operations(code_str: str) -> bool:
        """
        Check if the code contains boolean operations (and/or) that can be mutated with DeMorgan's laws.
        
        Args:
            code_str (str): The code to analyze
            
        Returns:
            bool: True if code contains boolean operations suitable for DeMorgan mutation
        """
        try:
            tree = ast.parse(code_str)
            
            # Walk through all nodes to find boolean operations
            for node in ast.walk(tree):
                # Check for BoolOp nodes (and/or operations)
                if isinstance(node, ast.BoolOp):
                    if isinstance(node.op, (ast.And, ast.Or)):
                        return True
                
                # Check for UnaryOp with Not that could be applied to boolean operations
                if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.Not):
                    if isinstance(node.operand, ast.BoolOp):
                        return True
                
            return False
            
        except (SyntaxError, ValueError):
            # If code can't be parsed, assume it's not suitable for mutation
            return False
    
    @staticmethod
    def filter_database_for_demorgan(llmtester):
        """
        Filter the database to only include entries that can be mutated with DeMorgan's laws.
        
        Args:
            llmtester: LLMConsistencyTester instance
            
        Returns:
            list: List of filtered document IDs suitable for DeMorgan mutation
        """
        suitable_docs = []
        
        # Get all documents from the database
        all_docs = list(llmtester.question_database.find({}))
        
        print(f"Checking {len(all_docs)} documents for DeMorgan mutation suitability...")
        
        for doc in all_docs:
            # Check if the solution contains boolean operations
            if 'full_sol' in doc and DeMorganPreConditionFilter.has_boolean_operations(doc['full_sol']):
                suitable_docs.append(doc['_id'])
        
        print(f"Found {len(suitable_docs)} documents suitable for DeMorgan mutation out of {len(all_docs)} total documents")
        
        return suitable_docs

In [ ]:
prompt_template = CodeInconsistencyPromptTemplate.zero_shot_prompt()
llmtester = LLMConsistencyTester("HumanEval_Input_Output")

# Test basic database connection and row loading
print(f"Database connection established")
total_docs = llmtester.question_database.count_documents({})
print(f"Total documents in database: {total_docs}")

# Show a sample document to verify structure
sample_doc = llmtester.question_database.find_one({})
if sample_doc:
    print(f"\nSample document structure:")
    print(f"Document ID: {sample_doc['_id']}")
    print(f"Has 'full_sol' field: {'full_sol' in sample_doc}")
    if 'full_sol' in sample_doc:
        print(f"Solution preview: {sample_doc['full_sol'][:100]}...")
else:
    print("No documents found in database")

In [ ]:
# Filter documents that can be mutated with DeMorgan's laws
demorgan_suitable_docs = DeMorganPreConditionFilter.filter_database_for_demorgan(llmtester)
print(f"\nDocuments suitable for DeMorgan mutation: {len(demorgan_suitable_docs)}")

In [ ]:
llm = Mistral()

In [ ]:
model_name = "mistral-small-2506"
mistral_results = os.path.join(proj_dir + '/results/code_inconsistencies/mistral')
os.makedirs(mistral_results, exist_ok=True)

In [ ]:
print(f"\nDocuments suitable for DeMorgan mutation: {len(demorgan_suitable_docs)}")
print(demorgan_suitable_docs)

In [ ]:
# Run DeMorgan mutation consistency test on filtered documents
if len(demorgan_suitable_docs) > 0:
    # Set up dynamic file naming like the original tester
    syntactic_mutation = "demorgan"
    prompt_type = "zero_shot"
    mutation_str = f"{syntactic_mutation}_mutation"
    
    output_file_path = f"{mistral_results}/{model_name}_{prompt_type}_{mutation_str}.csv"
    print(f"Results will be saved to: {output_file_path}")
    
    # MODIFICATION: Test only first 10 documents
    test_docs = demorgan_suitable_docs[:10]
    print(f"Running DeMorgan consistency test on {len(test_docs)} suitable documents (first 10)...")

    print('llmtester: ', llmtester)

    print()
    
    pass_count = DeMorganConsistencyTester.run_demorgan_consistency_test(
        llmtester=llmtester,
        llm=llm,
        suitable_doc_ids=test_docs,
        output_file_path=output_file_path
    )
    
    print(f"\nDeMorgan consistency test completed!")
    print(f"Results saved to: {output_file_path}")
else:
    print("No documents found suitable for DeMorgan mutation. Skipping test.")

## Analysis

This notebook tests LLM consistency using DeMorgan's law mutations, which transform boolean expressions:
- `A and B` → `not ((not A) or (not B))`
- `A or B` → `not ((not A) and (not B))`
- `not (A and B)` → `(not A) or (not B)`
- `not (A or B)` → `(not A) and (not B)`

The pre-condition filter ensures we only test on code that contains boolean operations suitable for these transformations.